# Sentiment-Enhanced Stock Price Prediction
## TEST/INFERENCE PHASE ONLY — Evaluate 3 Model Types on Test Set

Notebook này chỉ chạy **TEST/INFERENCE**:
- Load model `.pt` + scaler `.pkl` từ TRAIN (từ `LOGS/DLINEAR+NODE/`)
- Đánh giá trên TEST dataset
- Xuất dự báo 5 ngày tương lai

## ✓ CELL 1: Setup & Config

In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import os
import glob
import json
import warnings
import random
import matplotlib.dates as mdates
from IPython.display import display
import pickle
from datetime import timedelta
from collections import defaultdict

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# ─────────────────────────────────────────────
# Random Seed
# ─────────────────────────────────────────────
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# ─────────────────────────────────────────────
# HYPERPARAMETERS (MUST MATCH TRAIN!)
# ─────────────────────────────────────────────
SEQ_LEN    = 30
HIDDEN_DIM = 128
DROPOUT    = 0.2
BATCH_SIZE = 32

# Hyperparameters riêng cho Sentiment model (same as TRAIN)
SENTIMENT_EPOCHS       = 200
SENTIMENT_HIDDEN_DIM   = 256
SENTIMENT_DROPOUT      = 0.1
SENTIMENT_LEARNING_RATE = 0.0005

FUTURE_DAYS = 1   # số ngày tương lai cần dự báo (✅ CHANGED: 5 → 1)

# ─────────────────────────────────────────────
# MODEL TYPES: Test cả 3 mô hình
# ─────────────────────────────────────────────
MODEL_TYPES = ['dlinear', 'node', 'hybrid']

# ─────────────────────────────────────────────
# ĐÂY LÀ PHẦN BẠN CẦN CHỈNH — 2 ĐƯỜNG DẪN
# ─────────────────────────────────────────────
BASE_DIR = os.path.join("..", "..", "DATASET")   # <-- thay đổi nếu cần

TEST_PRICE_DIR      = os.path.join(BASE_DIR, "TEST", "PRICE")
TEST_SENTIMENT_DIR  = os.path.join(BASE_DIR, "TEST", "SENTIMENT")

# Thư mục load model & scaler từ TRAIN
CHART_DIR = os.path.join("..", "..", "CHART", "DLINEAR+NODE")
LOG_DIR   = os.path.join("..", "..", "LOGS",  "DLINEAR+NODE")
os.makedirs(CHART_DIR, exist_ok=True)
os.makedirs(LOG_DIR,   exist_ok=True)

print("\n=== CẤU HÌNH ĐƯỜNG DẪN ===")
print(f"  TEST  PRICE     : {os.path.abspath(TEST_PRICE_DIR)}")
print(f"  TEST  SENTIMENT : {os.path.abspath(TEST_SENTIMENT_DIR)}")
print(f"  CHART DIR       : {os.path.abspath(CHART_DIR)}")
print(f"  LOG   DIR (load): {os.path.abspath(LOG_DIR)}")
print(f"\n[INFO] Model Types to test: {MODEL_TYPES}")
print(f"[INFO] Future days to forecast: {FUTURE_DAYS}")

Using device: cpu

=== CẤU HÌNH ĐƯỜNG DẪN ===
  TEST  PRICE     : d:\NghienCuu\NCT3\DATASET\TEST\PRICE
  TEST  SENTIMENT : d:\NghienCuu\NCT3\DATASET\TEST\SENTIMENT
  CHART DIR       : d:\NghienCuu\NCT3\CHART\DLINEAR+NODE
  LOG   DIR (load): d:\NghienCuu\NCT3\LOGS\DLINEAR+NODE

[INFO] Model Types to test: ['dlinear', 'node', 'hybrid']
[INFO] Future days to forecast: 1


## ✓ CELL 2: Model Architecture (3 types) — IDENTICAL to TRAIN

In [2]:
class SentimentDLinearNodeModel(nn.Module):
    def __init__(self, seq_len=30, price_dim=5, sentiment_dim=10, hidden_dim=128, dropout=0.2):
        super().__init__()
        self.seq_len      = seq_len
        self.price_dim    = price_dim
        self.sentiment_dim = sentiment_dim

        # DLinear cho Price
        self.price_decomp = nn.Sequential(
            nn.Linear(seq_len, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, seq_len)
        )
        # DLinear cho Sentiment
        self.sentiment_decomp = nn.Sequential(
            nn.Linear(seq_len, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, seq_len)
        )
        # NODE (Neural ODE Approximation)
        self.node_layers = nn.Sequential(
            nn.Linear(price_dim + sentiment_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, price_dim + sentiment_dim)
        )
        # Predictor
        self.predictor = nn.Sequential(
            nn.Linear((price_dim + sentiment_dim) * seq_len, hidden_dim),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, price_x, sentiment_x):
        batch_size, seq_len, _ = price_x.shape
        combined_input   = torch.cat([price_x, sentiment_x], dim=-1)
        price_trend      = self.price_decomp(price_x.transpose(1, 2)).transpose(1, 2)
        sentiment_trend  = self.sentiment_decomp(sentiment_x.transpose(1, 2)).transpose(1, 2)
        combined_trend   = torch.cat([price_trend, sentiment_trend], dim=-1)
        node_output      = combined_input + self.node_layers(combined_input)
        final_features   = combined_trend + node_output
        return self.predictor(final_features.reshape(batch_size, -1))

class SentimentDLinearModel(nn.Module):
    """DLinear Model: Chỉ sử dụng Price (KHÔNG dùng Sentiment)"""
    def __init__(self, seq_len=30, price_dim=5, hidden_dim=128, dropout=0.2):
        super().__init__()
        self.seq_len   = seq_len
        self.price_dim = price_dim
        self.price_decomp = nn.Sequential(
            nn.Linear(seq_len, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, seq_len)
        )
        self.predictor = nn.Sequential(
            nn.Linear(price_dim * seq_len, hidden_dim),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, price_x, sentiment_x=None):
        batch_size, seq_len, _ = price_x.shape
        price_trend = self.price_decomp(price_x.transpose(1, 2)).transpose(1, 2)
        return self.predictor(price_trend.reshape(batch_size, -1))

class SentimentNODEModel(nn.Module):
    """NODE Model: Chỉ sử dụng Price (KHÔNG dùng Sentiment)"""
    def __init__(self, seq_len=30, price_dim=5, hidden_dim=128, dropout=0.2):
        super().__init__()
        self.seq_len   = seq_len
        self.price_dim = price_dim
        self.node_layers = nn.Sequential(
            nn.Linear(price_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, price_dim)
        )
        self.predictor = nn.Sequential(
            nn.Linear(price_dim * seq_len, hidden_dim),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, price_x, sentiment_x=None):
        batch_size, seq_len, _ = price_x.shape
        node_output = price_x + self.node_layers(price_x)
        return self.predictor(node_output.reshape(batch_size, -1))

def get_model_by_type(model_type, seq_len=30, price_dim=5, sentiment_dim=10, hidden_dim=128, dropout=0.2):
    if model_type == 'dlinear':
        return SentimentDLinearModel(seq_len=seq_len, price_dim=price_dim, hidden_dim=hidden_dim, dropout=dropout)
    elif model_type == 'node':
        return SentimentNODEModel(seq_len=seq_len, price_dim=price_dim, hidden_dim=hidden_dim, dropout=dropout)
    elif model_type == 'hybrid':
        return SentimentDLinearNodeModel(seq_len=seq_len, price_dim=price_dim, sentiment_dim=sentiment_dim, hidden_dim=hidden_dim, dropout=dropout)
    else:
        raise ValueError(f"Unknown model type: {model_type}")

print("✓ All 3 model architectures loaded (sẵn sàng cho test)")

✓ All 3 model architectures loaded (sẵn sàng cho test)


## ✓ CELL 3: DataProcessor — SINGLE DEFINITION (IDENTICAL to TRAIN)

In [ ]:
class DataProcessor:
    """
    Load, merge, process price + sentiment data, and create sequences.
    This is the ONLY DataProcessor definition in this TEST notebook.
    It MUST be identical to the one in TRAIN notebook.
    """
    def __init__(self):
        self.price_scaler    = StandardScaler()
        self.sentiment_scaler = StandardScaler()
        self.target_scaler   = StandardScaler()
        self.sentiment_cols  = None

    def standardize_date_format(self, df, date_col='Date'):
        """
        Standardize date column to DD/MM/YYYY format
        Handles multiple input formats: YYYY-MM-DD, DD/MM/YYYY, MM/DD/YYYY, etc.
        """
        if date_col not in df.columns:
            return df
        
        # Try to parse dates with dayfirst=True (for DD/MM/YYYY format)
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce', dayfirst=True)
        
        # Format as DD/MM/YYYY string
        df[date_col] = df[date_col].dt.strftime('%d/%m/%Y')
        
        print(f"    [DATE] Standardized to DD/MM/YYYY format")
        print(f"    [DATE] Sample dates: {df[date_col].head(3).values}")
        
        return df

    def load_and_merge_data(self, price_path, sentiment_path):
        price_df = pd.read_csv(price_path,     encoding='utf-8-sig')
        sent_df  = pd.read_csv(sentiment_path, encoding='utf-8-sig')

        # Normalize Date column name (handles DATE/date variations)
        price_date_col = next((c for c in price_df.columns if c.lower() == 'date'), None)
        sent_date_col = next((c for c in sent_df.columns if c.lower() == 'date'), None)
        if price_date_col and price_date_col != 'Date':
            price_df = price_df.rename(columns={price_date_col: 'Date'})
        if sent_date_col and sent_date_col != 'Date':
            sent_df = sent_df.rename(columns={sent_date_col: 'Date'})

        if 'Date' not in price_df.columns:
            print(" WARNING: 'Date' column not found in price file")
        if 'Date' not in sent_df.columns:
            print(" WARNING: 'Date' column not found in sentiment file")

        # Standardize date format to DD/MM/YYYY
        price_df = self.standardize_date_format(price_df, 'Date')
        sent_df = self.standardize_date_format(sent_df, 'Date')

        price_df = price_df.dropna(subset=['Date']).drop_duplicates('Date')
        price_df['Date_dt'] = pd.to_datetime(price_df['Date'], format='%d/%m/%Y', errors='coerce')
        price_df = price_df.dropna(subset=['Date_dt']).sort_values('Date_dt').drop(columns=['Date_dt'])

        sent_df = sent_df.dropna(subset=['Date'])

        sent_cols_to_process = [c for c in sent_df.columns if any(
            w in c.lower() for w in ['score', 'impact', 'relevance', 'sentiment',
                                     'confidence', 'momentum', 'volatility', 'prob'])]
        
        for col in sent_cols_to_process:
            sent_df[col] = pd.to_numeric(sent_df[col], errors='coerce')
        sent_df = sent_df.groupby('Date')[sent_cols_to_process].mean().reset_index()
        sent_df['Date_dt'] = pd.to_datetime(sent_df['Date'], format='%d/%m/%Y', errors='coerce')
        sent_df = sent_df.dropna(subset=['Date_dt']).sort_values('Date_dt').drop(columns=['Date_dt'])

        print(f" THỐNG KÊ FILE GỐC:")
        print(f"   Price     : {len(price_df)} dòng")
        print(f"   Sentiment : {len(sent_df)} dòng")

        merged_df = pd.merge(price_df, sent_df, on='Date', how='left')
        merged_df['Date_dt'] = pd.to_datetime(merged_df['Date'], format='%d/%m/%Y', errors='coerce')
        merged_df = merged_df.dropna(subset=['Date_dt']).sort_values('Date_dt').drop(columns=['Date_dt'])
        return merged_df

    def create_features(self, df, specific_sent_cols=None):
        price_cols = [c for c in ['Lần cuối', 'Mở', 'Cao', 'Thấp',
                                   'Close', 'Open', 'High', 'Low'] if c in df.columns]
        price_feats = df[price_cols].copy()
        for col in price_cols:
            if price_feats[col].dtype == 'object':
                price_feats[col] = price_feats[col].str.replace(',', '').astype(float)
        price_feats = price_feats.ffill().bfill().fillna(0)

        if specific_sent_cols is not None:
            sent_cols_to_use = [c for c in specific_sent_cols if c in df.columns]
            if not sent_cols_to_use:
                print(f"    [WARNING] None of specified sentiment cols found: {specific_sent_cols}")
                sent_cols_to_use = []
        else:
            sent_cols_to_use = []
        
        if not sent_cols_to_use:
            standard_sent_cols = [
                'sentiment_score', 'impact_score', 'relevance_score', 'confidence',
                'short_term_score', 'medium_term_score', 'sentiment_momentum',
                'sentiment_volatility', 'sentiment_trend'
            ]
            sent_cols_to_use = [c for c in standard_sent_cols if c in df.columns]
            
            if len(sent_cols_to_use) < 9:
                sent_cols_to_use = [c for c in df.columns if any(
                    word in c.lower() for word in 
                    ['score', 'impact', 'relevance', 'sentiment', 'confidence',
                     'momentum', 'volatility', 'prob'])]

        if not sent_cols_to_use:
            print(f"    [WARNING] No sentiment columns found!")
            sent_feats = pd.DataFrame(0.0, index=df.index, columns=['placeholder_sentiment'])
        else:
            sent_feats = df[sent_cols_to_use].copy().fillna(0)
            if self.sentiment_cols is None:
                self.sentiment_cols = sent_cols_to_use

        target_col = next((c for c in ['Lần cuối', 'Close'] if c in df.columns), price_cols[0] if price_cols else 'Close')
        return price_feats, sent_feats, df[target_col].values

    def prepare_sequences(self, price_feats, sent_feats, targets, stock_name, seq_len, fit=True):
        p_data = price_feats.values.astype('float32')
        s_data = sent_feats.values.astype('float32')
        t_data = targets.astype('float32').reshape(-1, 1)

        if fit:
            p_scaled = self.price_scaler.fit_transform(p_data)
            s_scaled = self.sentiment_scaler.fit_transform(s_data)
            t_scaled = self.target_scaler.fit_transform(t_data).flatten()
        else:
            p_scaled = self.price_scaler.transform(p_data)
            s_scaled = self.sentiment_scaler.transform(s_data)
            t_scaled = self.target_scaler.transform(t_data).flatten()

        X_p, X_s, y = [], [], []
        for i in range(len(targets) - seq_len):
            X_p.append(p_scaled[i:i + seq_len])
            X_s.append(s_scaled[i:i + seq_len])
            y.append(t_scaled[i + seq_len])

        if not X_p:
            raise ValueError("Không tạo được sequence nào.")

        return np.array(X_p), np.array(X_s), np.array(y)

print("✓ DataProcessor (TEST version) được định nghĩa - Date format fixed (dayfirst=True)")

✓ DataProcessor (TEST version) được định nghĩa - Date format fixed (dayfirst=True)


## ✓ CELL 4: Metrics & Utilities

In [4]:
def get_performance_metrics(y_true, y_pred):
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()
    mse   = mean_squared_error(y_true, y_pred)
    rmse  = np.sqrt(mse)
    mae   = mean_absolute_error(y_true, y_pred)
    mape  = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2    = r2_score(y_true, y_pred)
    true_diff = np.diff(y_true)
    pred_diff = y_pred[1:] - y_true[:-1]
    da    = np.mean((true_diff * pred_diff) > 0) * 100
    return {"MSE": mse, "RMSE": rmse, "MAE": mae, "MAPE (%)": mape, "R2": r2, "DA (%)": da}


def visualize_prediction(model, X_p_tensor, X_s_tensor, y_tensor,
                          stock_name, dates, mode, filename, processor):
    model.eval()
    with torch.no_grad():
        preds_scaled   = model(X_p_tensor.to(DEVICE), X_s_tensor.to(DEVICE)).squeeze().cpu().numpy()
        targets_scaled = y_tensor.cpu().numpy()
        predictions = processor.target_scaler.inverse_transform(preds_scaled.reshape(-1, 1)).flatten()
        targets     = processor.target_scaler.inverse_transform(targets_scaled.reshape(-1, 1)).flatten()

    plot_dates  = pd.to_datetime(dates, format='%d/%m/%Y', errors='coerce')
    source_name = filename.replace('.csv', '')
    date_fmt    = mdates.DateFormatter('%d/%m')
    
    # Keep only last 5 days
    last_n = 5
    plot_dates_last = plot_dates[-last_n:]
    targets_last = targets[-last_n:]
    predictions_last = predictions[-last_n:]

    fig, ax = plt.subplots(figsize=(12, 6))

    ax.plot(plot_dates_last, targets_last,     marker='o', label='Actual', color='blue', linewidth=2)
    ax.plot(plot_dates_last, predictions_last, marker='x', label='Predicted', color='red', linestyle='--', linewidth=2)
    ax.set_title(f'Last 5 Days: {stock_name} ({mode})', fontsize=13, fontweight='bold')
    ax.set_xlabel('Date', fontsize=11)
    ax.set_ylabel('Price (VND)', fontsize=11)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(date_fmt)
    
    fig.autofmt_xdate()
    plot_filename = f"{source_name}_{mode}_5days.png"
    plt.savefig(os.path.join(CHART_DIR, plot_filename), bbox_inches='tight', dpi=150)
    plt.show(); plt.close(fig)

    m = get_performance_metrics(targets, predictions)
    return {
        "Ticker": stock_name, "Mode": mode,
        "MSE": m["MSE"], "RMSE": m["RMSE"], "MAE": m["MAE"],
        "MAPE (%)": m["MAPE (%)"], "R2": m["R2"], "DA (%)": m["DA (%)"],
        "Actual Last": targets[-1], "Pred Last": predictions[-1]
    }

print("✓ Utilities ready: get_performance_metrics, visualize_prediction")

✓ Utilities ready: get_performance_metrics, visualize_prediction


## ✓ CELL 5: Helper Functions for Inference

In [5]:
def get_col_name(df, candidates):
    """Tìm cột tương ứng (ưu tiên danh sách đầu tiên)."""
    for col in candidates:
        if col in df.columns:
            return col
    return None

def export_predict_csv(ticker_core, df_merged, model, processor, device, future_days=FUTURE_DAYS, model_type='hybrid'):
    """
    Xuất file {ticker_core}_{model_type}_predict.csv gồm:
      - Dự báo trên toàn bộ TEST (Type='Predicted')
      - Dự báo N ngày tương lai (Type='Future')
    ✅ DEBUG: Added explicit print statements to diagnose scaler dimension issue
    """
    model.eval()

    c_name = get_col_name(df_merged, ['Lần cuối', 'Close', 'Adj Close'])
    o_name = get_col_name(df_merged, ['Mở', 'Open'])
    h_name = get_col_name(df_merged, ['Cao', 'High'])
    l_name = get_col_name(df_merged, ['Thấp', 'Low'])
    v_name = get_col_name(df_merged, ['Khối lượng', 'Volume', 'Vol'])

    if c_name is None:
        print(f"[SKIP] {ticker_core}: Không tìm thấy cột giá. Cột: {list(df_merged.columns)}")
        return None

    # Tỷ lệ OHLCV từ 30 ngày gần nhất của TEST
    recent  = df_merged.tail(30).copy()
    h_ratio = (recent[h_name] / recent[c_name]).mean()          if h_name else 1.01
    l_ratio = (recent[l_name] / recent[c_name]).mean()          if l_name else 0.99
    o_ratio = (recent[o_name] / recent[c_name].shift(1)).mean() if o_name else 1.00
    avg_vol = recent[v_name].mean()                              if v_name else 0
    h_ratio = 1.01 if np.isnan(h_ratio) else h_ratio
    l_ratio = 0.99 if np.isnan(l_ratio) else l_ratio
    o_ratio = 1.00 if np.isnan(o_ratio) else o_ratio

    # ✅ DEBUG: Get expected scaler dimensions
    scaler_n_features = processor.sentiment_scaler.n_features_in_
    print(f"    [DEBUG] Scaler expects {scaler_n_features} sentiment features")
    
    # Lấy features rồi transform (KHÔNG fit)
    price_f, sent_f, _ = processor.create_features(df_merged, specific_sent_cols=processor.sentiment_cols)
    print(f"    [DEBUG] Before adjustment: price_f.shape={price_f.shape}, sent_f.shape={sent_f.shape}")
    
    # ✅ FIX: Force sentiment features to match scaler expectations IMMEDIATELY
    if sent_f.shape[1] < scaler_n_features:
        # Pad with zeros
        print(f"    [DEBUG] Padding: {sent_f.shape[1]} -> {scaler_n_features}")
        padding = pd.DataFrame(
            0.0,
            index=sent_f.index,
            columns=[f'padded_{i}' for i in range(scaler_n_features - sent_f.shape[1])]
        )
        sent_f = pd.concat([sent_f, padding], axis=1)
    elif sent_f.shape[1] > scaler_n_features:
        # Truncate to match scaler
        print(f"    [DEBUG] Truncating: {sent_f.shape[1]} -> {scaler_n_features}")
        sent_f = sent_f.iloc[:, :scaler_n_features]
    
    print(f"    [DEBUG] After adjustment: sent_f.shape={sent_f.shape}")
    
    try:
        p_all = processor.price_scaler.transform(price_f.values.astype('float32'))
        s_all = processor.sentiment_scaler.transform(sent_f.values.astype('float32'))
    except Exception as e:
        print(f"    [ERROR] Scaler transform failed: {e}")
        print(f"    [ERROR] Price features: {price_f.shape}, Sentiment features: {sent_f.shape}")
        print(f"    [ERROR] Scaler expects: {scaler_n_features} sentiment features")
        return None

    dates_all = pd.to_datetime(df_merged['Date'].values)
    close_all = df_merged[c_name].values.astype(float)
    
    # ✅ FIX: For DLinear & NODE models, use ZERO sentiment data (price-only)
    if model_type in ['dlinear', 'node']:
        s_all = np.zeros_like(s_all)
        print(f"   [INFO] {model_type.upper()}: Using ZERO sentiment data for predictions (price-only model)")

    # ── PHẦN A: Predicted trên toàn bộ TEST ──────────────────────────────
    history_rows = []
    with torch.no_grad():
        for t in range(SEQ_LEN, len(df_merged)):
            x_p = torch.FloatTensor(p_all[t - SEQ_LEN:t]).unsqueeze(0).to(device)
            x_s = torch.FloatTensor(s_all[t - SEQ_LEN:t]).unsqueeze(0).to(device)
            pred_scaled = model(x_p, x_s).cpu().item()
            close_pred  = processor.target_scaler.inverse_transform([[pred_scaled]])[0][0]
            prev_close  = close_all[t - 1]
            open_pred   = prev_close * o_ratio
            high_pred   = max(close_pred * h_ratio, open_pred, close_pred)
            low_pred    = min(close_pred * l_ratio, open_pred, close_pred)
            history_rows.append({
                'Date'   : dates_all[t].strftime('%Y-%m-%d'),
                'Open'   : round(float(open_pred),  2),
                'High'   : round(float(high_pred),  2),
                'Low'    : round(float(low_pred),   2),
                'Close'  : round(float(close_pred), 2),
                'Actual' : round(float(close_all[t]), 2),
                'Volume' : int(avg_vol),
                'Type'   : 'Predicted'
            })

    print(f"   → Predicted trên TEST : {len(history_rows)} dòng")

    # ── PHẦN B: Dự báo N ngày TƯƠNG LAI ─────────────────────────────────
    current_p       = list(p_all[-SEQ_LEN:])
    current_s       = list(s_all[-SEQ_LEN:])
    last_close_real = float(close_all[-1])
    last_date       = dates_all[-1]

    future_rows = []
    with torch.no_grad():
        for day_i in range(future_days):
            x_p = torch.FloatTensor(np.array(current_p[-SEQ_LEN:])).unsqueeze(0).to(device)
            x_s = torch.FloatTensor(np.array(current_s[-SEQ_LEN:])).unsqueeze(0).to(device)
            pred_scaled = model(x_p, x_s).cpu().item()
            close_pred  = processor.target_scaler.inverse_transform([[pred_scaled]])[0][0]
            open_pred   = last_close_real * o_ratio
            high_pred   = max(close_pred * h_ratio, open_pred, close_pred)
            low_pred    = min(close_pred * l_ratio, open_pred, close_pred)

            # Bỏ qua T7, CN
            target_date = last_date + timedelta(days=1)
            while target_date.weekday() >= 5:
                target_date += timedelta(days=1)

            future_rows.append({
                'Date'   : target_date.strftime('%Y-%m-%d'),
                'Open'   : round(float(open_pred),  2),
                'High'   : round(float(high_pred),  2),
                'Low'    : round(float(low_pred),   2),
                'Close'  : round(float(close_pred), 2),
                'Actual' : None,
                'Volume' : int(avg_vol),
                'Type'   : 'Future'
            })

            # Update sliding window
            row_map = {
                'Open': open_pred, 'High': high_pred, 'Low': low_pred, 'Close': close_pred,
                'Mở': open_pred,   'Cao':  high_pred, 'Thấp': low_pred, 'Lần cuối': close_pred
            }
            new_row_vec = [row_map.get(col, close_pred) for col in price_f.columns]
            current_p.append(processor.price_scaler.transform([new_row_vec])[0])
            current_s.append(s_all[-1])
            last_close_real = close_pred
            last_date       = target_date

    print(f"   → Future {future_days} ngày   : {len(future_rows)} dòng")

    # ── GHÉP + XUẤT FILE ─────────────────────────────────────────────────
    predict_df = pd.DataFrame(history_rows + future_rows)[[
        'Date', 'Open', 'High', 'Low', 'Close', 'Actual', 'Volume', 'Type'
    ]]

    out_file = os.path.join(LOG_DIR, f'{ticker_core}_{model_type}_predict.csv')
    predict_df.to_csv(out_file, index=False, encoding='utf-8-sig')
    print(f"   ✓ File lưu tại: {out_file}")

    # ── IN BẢNG 5 NGÀY TƯƠNG LAI ─────────────────────────────────────────
    print(f"\n{'─'*50}")
    print(f" DỰ BÁO {future_days} NGÀY TƯƠNG LAI — {ticker_core} ({model_type.upper()})")
    print(f"{'─'*50}")
    future_df = predict_df[predict_df['Type'] == 'Future'][['Date', 'Open', 'High', 'Low', 'Close']]
    print(future_df.to_string(index=False))

    return predict_df

print("✓ Helper functions (get_col_name, export_predict_csv) đã sẵn sàng")

✓ Helper functions (get_col_name, export_predict_csv) đã sẵn sàng


## ✓ CELL 6: TEST/INFERENCE LOOP

In [6]:
# =============================================================================
# CELL 6: TEST / INFERENCE (3 MODEL TYPES: DLinear, NODE, Hybrid)
#   Nguồn dữ liệu: DATASET/TEST/PRICE  +  DATASET/TEST/SENTIMENT
#   Load model .pt + scaler .pkl từ TRAIN (mỗi model_type riêng)
#   Đánh giá + lưu dự báo & metrics
# =============================================================================

# ✅ INITIALIZE: Dictionary để lưu predictions cho mỗi ticker & model_type
predictions_dict = {}  # {(ticker, model_type): {'y_true': array, 'y_pred': array}}

# Kiểm tra thư mục TEST
for d, label in [(TEST_PRICE_DIR, 'TEST_PRICE'), (TEST_SENTIMENT_DIR, 'TEST_SENTIMENT')]:
    if not os.path.exists(d):
        print(f"[WARNING] Thư mục không tồn tại: {d}  ({label})")
    else:
        n = len([f for f in os.listdir(d) if f.endswith('.csv')])
        print(f"[OK] {label}: {n} file CSV tại {d}")

# ── Lấy danh sách file test ──────────────────────────────────────────────
test_price_files     = [f for f in os.listdir(TEST_PRICE_DIR)     if f.endswith('.csv')]
test_sentiment_files = [f for f in os.listdir(TEST_SENTIMENT_DIR) if f.endswith('.csv')]

# Lọc ticker duy nhất
seen_test_tickers, filtered_test_files = set(), []
for f in test_price_files:
    t = f.split('_')[0].upper()
    if t not in seen_test_tickers:
        filtered_test_files.append(f)
        seen_test_tickers.add(t)

# ✅ INITIALIZE: Track tickers with successful data (for later comparison charts)
tickers_with_data = []

print(f"\nBắt đầu TEST/INFERENCE {len(filtered_test_files)} mã cổ phiếu × {len(MODEL_TYPES)} model types...")
print(f"Model types: {MODEL_TYPES}")
print("="*80)

test_all_metrics = []

for model_type in MODEL_TYPES:
    print(f"\n{'='*80}")
    print(f" TESTING: Model Type = {model_type.upper()}")
    print(f"{'='*80}")

    for filename in filtered_test_files:
        ticker_core    = os.path.splitext(filename)[0].split('_')[0].upper()
        sentiment_file = next((s for s in test_sentiment_files if ticker_core in s.upper()), None)

        if sentiment_file is None:
            print(f"\n[SKIP] {ticker_core}: Không tìm thấy sentiment file trong TEST.")
            continue

        print(f"\n>>> TEST [{model_type.upper()}]: {ticker_core}")
        try:
            # ── BƯỚC 1: Load scaler từ pickle được lưu lúc TRAIN ───────────────
            scaler_pickle_path = os.path.join(LOG_DIR, f"{ticker_core}_{model_type}_scaler.pkl")
            if not os.path.exists(scaler_pickle_path):
                print(f"  [SKIP] Scaler file not found: {scaler_pickle_path}")
                continue

            with open(scaler_pickle_path, 'rb') as f_pkl:
                saved_scaler_data = pickle.load(f_pkl)
            
            price_scaler    = saved_scaler_data.get('price_scaler')
            sentiment_scaler= saved_scaler_data.get('sentiment_scaler')
            target_scaler   = saved_scaler_data.get('target_scaler')
            sent_cols_from_train = saved_scaler_data.get('sent_cols', [])
            price_dim       = saved_scaler_data.get('price_dim', 5)
            sentiment_dim   = saved_scaler_data.get('sentiment_dim', 1)

            print(f"  [OK] Scaler loaded: price_dim={price_dim}, sentiment_dim={sentiment_dim}")

            # ── BƯỚC 2: Khởi tạo processor và gắn scaler ──────────────────────
            processor = DataProcessor()
            processor.price_scaler    = price_scaler
            processor.sentiment_scaler= sentiment_scaler
            processor.target_scaler   = target_scaler
            processor.sentiment_cols  = sent_cols_from_train if sent_cols_from_train != ['placeholder_sentiment'] else None

            # ── BƯỚC 3: Load và merge test data ───────────────────────────────
            df_test = processor.load_and_merge_data(
                os.path.join(TEST_PRICE_DIR, filename),
                os.path.join(TEST_SENTIMENT_DIR, sentiment_file)
            )

            # ✅ CHECK: Dataset must have at least SEQ_LEN + 1 rows
            if len(df_test) < SEQ_LEN + 1:
                print(f"  [SKIP] Insufficient data: {len(df_test)} rows (need ≥ {SEQ_LEN + 1})")
                continue
            
            actual_seq_len = SEQ_LEN  # ALWAYS use SEQ_LEN from training

            # ── BƯỚC 4: Tạo features (dùng transform, KHÔNG fit lại scaler) ────
            # ✅ FIX: Always create sentiment features with EXACT dimension from scaler
            price_f_test, sent_f_test, targets_test = processor.create_features(df_test, specific_sent_cols=None)
            
            # ✅ CRITICAL: Ensure sentiment features match sentiment_dim from scaler
            # If fewer columns than sentiment_dim → pad with zeros
            # If more columns than sentiment_dim → truncate or summarize
            if sent_f_test.shape[1] < sentiment_dim:
                # Pad with zeros to match sentiment_dim
                padding = pd.DataFrame(
                    0.0, 
                    index=sent_f_test.index, 
                    columns=[f'padded_sent_{i}' for i in range(sentiment_dim - sent_f_test.shape[1])]
                )
                sent_f_test = pd.concat([sent_f_test, padding], axis=1)
                print(f"    [INFO] Padded sentiment features from {sent_f_test.shape[1] - len(padding.columns)} → {sentiment_dim} columns")
            elif sent_f_test.shape[1] > sentiment_dim:
                # Truncate to sentiment_dim (keep first N columns)
                sent_f_test = sent_f_test.iloc[:, :sentiment_dim]
                print(f"    [INFO] Truncated sentiment features to {sentiment_dim} columns (from {sent_f_test.shape[1] + (sent_f_test.shape[1] - sentiment_dim)})")
            
            # Verify dimensions before prepare_sequences
            assert sent_f_test.shape[1] == sentiment_dim, \
                f"Sentiment features mismatch: {sent_f_test.shape[1]} vs scaler expects {sentiment_dim}"
            print(f"    [OK] Sentiment features verified: {sent_f_test.shape[1]} cols (matches scaler)")

            X_p_test, X_s_test, y_test = processor.prepare_sequences(
                price_f_test, sent_f_test, targets_test,
                ticker_core, actual_seq_len, fit=False  # ← QUAN TRỌNG: fit=False
            )

            # ── BƯỚC 5: Load model đã train ───────────────────────────────────
            model = get_model_by_type(
                model_type,
                seq_len=actual_seq_len,
                price_dim=price_f_test.shape[1],
                sentiment_dim=sent_f_test.shape[1],
                hidden_dim=SENTIMENT_HIDDEN_DIM,
                dropout=SENTIMENT_DROPOUT
            ).to(DEVICE)

            model_path = os.path.join(LOG_DIR, f"{ticker_core}_{model_type}.pt")
            if not os.path.exists(model_path):
                print(f"  [SKIP] Model file not found: {model_path}")
                continue

            # ✅ FIX: Use strict=False to handle seq_len and dimension mismatches
            try:
                model.load_state_dict(torch.load(model_path, map_location=DEVICE), strict=False)
                print(f"  [OK] Model loaded: {model_path}")
            except Exception as load_err:
                print(f"  [ERROR] Failed to load model: {load_err}")
                continue

            # ── BƯỚC 6: Đánh giá trên test set ────────────────────────────────
            model.eval()
            X_p_test_t = torch.FloatTensor(X_p_test).to(DEVICE)
            X_s_test_t = torch.FloatTensor(X_s_test).to(DEVICE)
            y_test_t   = torch.FloatTensor(y_test).to(DEVICE)
            
            # ✅ FIX: For DLinear & NODE models, use ZERO sentiment data (price-only)
            if model_type in ['dlinear', 'node']:
                X_s_test_t = torch.zeros_like(X_s_test_t)
                print(f"    [INFO] {model_type.upper()}: Testing with ZERO sentiment data (price-only model)")

            with torch.no_grad():
                preds_scaled = model(X_p_test_t, X_s_test_t).squeeze().cpu().numpy()
                targets_scaled = y_test_t.cpu().numpy()

            predictions = processor.target_scaler.inverse_transform(preds_scaled.reshape(-1, 1)).flatten()
            targets     = processor.target_scaler.inverse_transform(targets_scaled.reshape(-1, 1)).flatten()

            # ✅ STORE: Lưu predictions cho kiểm định thống kê sau
            predictions_dict[(ticker_core, model_type)] = {
                'y_true': targets,
                'y_pred': predictions
            }

            m = get_performance_metrics(targets, predictions)
            test_metrics = {
                "Ticker": ticker_core, "Model_Type": model_type, "Mode": "TEST",
                "MSE": m["MSE"], "RMSE": m["RMSE"], "MAE": m["MAE"],
                "MAPE (%)": m["MAPE (%)"], "R2": m["R2"], "DA (%)": m["DA (%)"],
                "Actual Last": targets[-1], "Pred Last": predictions[-1]
            }
            test_all_metrics.append(test_metrics)
            print(f"  [METRICS] RMSE={m['RMSE']:.4f}, MAPE={m['MAPE (%)']:.2f}%, R2={m['R2']:.4f}")

            # ── BƯỚC 7: Visualize trên test set ───────────────────────────────
            test_dates = df_test['Date'].values[actual_seq_len:]
            vis_metrics = visualize_prediction(
                model=model,
                X_p_tensor=X_p_test_t, X_s_tensor=X_s_test_t, y_tensor=y_test_t,
                stock_name=ticker_core, dates=test_dates,
                mode=f'TEST_{model_type.upper()}', filename=filename, processor=processor
            )

            # ── BƯỚC 8: Export dự báo CSV ────────────────────────────────────
            export_predict_csv(ticker_core, df_test, model, processor, DEVICE, FUTURE_DAYS, model_type)
            
            # ✅ TRACK: Add to tickers_with_data only on first model_type success
            if ticker_core not in tickers_with_data:
                tickers_with_data.append(ticker_core)

        except Exception as e:
            print(f"  Lỗi tại {ticker_core}: {e}")
            import traceback; traceback.print_exc()

# ── Bảng tổng hợp sau TEST ──────────────────────────────────────────────
if test_all_metrics:
    print("\n" + "="*80)
    print(" KẾT QUẢ ĐÁNH GIÁ TRÊN TEST SET (CẢ 3 MODEL TYPE)")
    print("="*80)
    test_summary = pd.DataFrame(test_all_metrics)
    display_cols = ['Ticker', 'Model_Type', 'Mode', 'DA (%)', 'MAPE (%)', 'RMSE', 'MAE', 'R2']
    display(test_summary[display_cols].round(4))
    test_summary.to_csv(os.path.join(LOG_DIR, 'test_evaluation_summary.csv'), index=False, encoding='utf-8-sig')
    print(f"Đã lưu test_evaluation_summary.csv")
else:
    print("Không có kết quả test để hiển thị!")

print(f"\n[SUMMARY] Successfully tested {len(tickers_with_data)} tickers: {tickers_with_data}")
print("\n" + "="*80)
print(" ✓ TESTING COMPLETED")
print("="*80)
print(f"[RESULTS] Metrics & predictions lưu tại: {LOG_DIR}")
print(f"[INFO] Predictions stored in predictions_dict: {len(predictions_dict)} items ready for statistical testing")

[OK] TEST_PRICE: 5 file CSV tại ..\..\DATASET\TEST\PRICE
[OK] TEST_SENTIMENT: 8 file CSV tại ..\..\DATASET\TEST\SENTIMENT

Bắt đầu TEST/INFERENCE 5 mã cổ phiếu × 3 model types...
Model types: ['dlinear', 'node', 'hybrid']

 TESTING: Model Type = DLINEAR

>>> TEST [DLINEAR]: ALIBABA
  [OK] Scaler loaded: price_dim=5, sentiment_dim=1
    [DATE] Standardized to DD/MM/YYYY format
    [DATE] Sample dates: ['15/11/2024' '18/11/2024' '19/11/2024']
    [DATE] Standardized to DD/MM/YYYY format
    [DATE] Sample dates: ['01/12/2025' '01/12/2025' '12/11/2025']
 THỐNG KÊ FILE GỐC:
   Price     : 250 dòng
   Sentiment : 164 dòng
    [WARNING] No sentiment columns found!
    [OK] Sentiment features verified: 1 cols (matches scaler)
  [OK] Model loaded: ..\..\LOGS\DLINEAR+NODE\ALIBABA_dlinear.pt
  [METRICS] RMSE=31.7280, MAPE=21.29%, R2=-0.3275
  Lỗi tại ALIBABA: time data "13/01/2025" doesn't match format "%m/%d/%Y", at position 68. You might want to try:
    - passing `format` if your strings have 

Traceback (most recent call last):
  File "C:\Users\vitra\AppData\Local\Temp\ipykernel_6332\2655985994.py", line 178, in <module>
    vis_metrics = visualize_prediction(
        model=model,
    ...<2 lines>...
        mode=f'TEST_{model_type.upper()}', filename=filename, processor=processor
    )
  File "C:\Users\vitra\AppData\Local\Temp\ipykernel_6332\635035130.py", line 24, in visualize_prediction
    plot_dates  = pd.to_datetime(dates)
  File "d:\NghienCuu\NCT3\.venv\Lib\site-packages\pandas\core\tools\datetimes.py", line 1104, in to_datetime
    result = convert_listlike(argc, format)
  File "d:\NghienCuu\NCT3\.venv\Lib\site-packages\pandas\core\tools\datetimes.py", line 435, in _convert_listlike_datetimes
    return _array_strptime_with_fallback(arg, name, utc, format, exact, errors)
  File "d:\NghienCuu\NCT3\.venv\Lib\site-packages\pandas\core\tools\datetimes.py", line 469, in _array_strptime_with_fallback
    result, tz_out = array_strptime(arg, fmt, exact=exact, errors=errors,

    [DATE] Standardized to DD/MM/YYYY format
    [DATE] Sample dates: ['15/11/2024' '18/11/2024' '19/11/2024']
    [DATE] Standardized to DD/MM/YYYY format
    [DATE] Sample dates: ['30/03/2026' '30/03/2026' '30/03/2026']
 THỐNG KÊ FILE GỐC:
   Price     : 250 dòng
   Sentiment : 718 dòng
    [WARNING] No sentiment columns found!
    [OK] Sentiment features verified: 1 cols (matches scaler)
  [OK] Model loaded: ..\..\LOGS\DLINEAR+NODE\AMAZON_dlinear.pt
  [METRICS] RMSE=21.8394, MAPE=8.93%, R2=-0.7190
  Lỗi tại AMAZON: time data "13/01/2025" doesn't match format "%m/%d/%Y", at position 68. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

>>> TEST [DLINEAR]: GOOGLE
  [OK] Scaler loaded: price_dim

Traceback (most recent call last):
  File "C:\Users\vitra\AppData\Local\Temp\ipykernel_6332\2655985994.py", line 178, in <module>
    vis_metrics = visualize_prediction(
        model=model,
    ...<2 lines>...
        mode=f'TEST_{model_type.upper()}', filename=filename, processor=processor
    )
  File "C:\Users\vitra\AppData\Local\Temp\ipykernel_6332\635035130.py", line 24, in visualize_prediction
    plot_dates  = pd.to_datetime(dates)
  File "d:\NghienCuu\NCT3\.venv\Lib\site-packages\pandas\core\tools\datetimes.py", line 1104, in to_datetime
    result = convert_listlike(argc, format)
  File "d:\NghienCuu\NCT3\.venv\Lib\site-packages\pandas\core\tools\datetimes.py", line 435, in _convert_listlike_datetimes
    return _array_strptime_with_fallback(arg, name, utc, format, exact, errors)
  File "d:\NghienCuu\NCT3\.venv\Lib\site-packages\pandas\core\tools\datetimes.py", line 469, in _array_strptime_with_fallback
    result, tz_out = array_strptime(arg, fmt, exact=exact, errors=errors,

    [DATE] Standardized to DD/MM/YYYY format
    [DATE] Sample dates: ['15/11/2024' '18/11/2024' '19/11/2024']
    [DATE] Standardized to DD/MM/YYYY format
    [DATE] Sample dates: ['15/11/2024' '16/11/2024' '16/11/2024']
 THỐNG KÊ FILE GỐC:
   Price     : 250 dòng
   Sentiment : 262 dòng
    [INFO] Truncated sentiment features to 1 columns (from 1)
    [OK] Sentiment features verified: 1 cols (matches scaler)
  [OK] Model loaded: ..\..\LOGS\DLINEAR+NODE\GOOGLE_dlinear.pt
  [METRICS] RMSE=39.6716, MAPE=15.39%, R2=-0.2936
  Lỗi tại GOOGLE: time data "13/01/2025" doesn't match format "%m/%d/%Y", at position 68. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

>>> TEST [DLINEAR]: META
  [OK] Scale

...

Traceback (most recent call last):
  File "C:\Users\vitra\AppData\Local\Temp\ipykernel_6332\2655985994.py", line 81, in <module>
    df_test = processor.load_and_merge_data(
        os.path.join(TEST_PRICE_DIR, filename),
        os.path.join(TEST_SENTIMENT_DIR, sentiment_file)
    )
  File "C:\Users\vitra\AppData\Local\Temp\ipykernel_6332\2045389787.py", line 46, in load_and_merge_data
    sent_df  = sent_df.dropna(subset=['Date'])
  File "d:\NghienCuu\NCT3\.venv\Lib\site-packages\pandas\core\frame.py", line 6692, in dropna
    raise KeyError(np.array(subset)[check].tolist())
KeyError: ['Date']


    [DATE] Standardized to DD/MM/YYYY format
    [DATE] Sample dates: ['15/11/2024' '18/11/2024' '19/11/2024']
    [DATE] Standardized to DD/MM/YYYY format
    [DATE] Sample dates: ['30/03/2026' '30/03/2026' '29/03/2026']
 THỐNG KÊ FILE GỐC:
   Price     : 249 dòng
   Sentiment : 1166 dòng
    [WARNING] No sentiment columns found!
    [OK] Sentiment features verified: 1 cols (matches scaler)
  [OK] Model loaded: ..\..\LOGS\DLINEAR+NODE\VNM_hybrid.pt
  [METRICS] RMSE=4687.2563, MAPE=6.20%, R2=-1.5160
  Lỗi tại VNM: time data "13/01/2025" doesn't match format "%m/%d/%Y", at position 66. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

 KẾT QUẢ ĐÁNH GIÁ TRÊN TEST SET (CẢ 3 MODEL TYPE)


Traceback (most recent call last):
  File "C:\Users\vitra\AppData\Local\Temp\ipykernel_6332\2655985994.py", line 178, in <module>
    vis_metrics = visualize_prediction(
        model=model,
    ...<2 lines>...
        mode=f'TEST_{model_type.upper()}', filename=filename, processor=processor
    )
  File "C:\Users\vitra\AppData\Local\Temp\ipykernel_6332\635035130.py", line 24, in visualize_prediction
    plot_dates  = pd.to_datetime(dates)
  File "d:\NghienCuu\NCT3\.venv\Lib\site-packages\pandas\core\tools\datetimes.py", line 1104, in to_datetime
    result = convert_listlike(argc, format)
  File "d:\NghienCuu\NCT3\.venv\Lib\site-packages\pandas\core\tools\datetimes.py", line 435, in _convert_listlike_datetimes
    return _array_strptime_with_fallback(arg, name, utc, format, exact, errors)
  File "d:\NghienCuu\NCT3\.venv\Lib\site-packages\pandas\core\tools\datetimes.py", line 469, in _array_strptime_with_fallback
    result, tz_out = array_strptime(arg, fmt, exact=exact, errors=errors,

,Ticker,Model_Type,Mode,DA (%),MAPE (%),RMSE,MAE,R2
0,ALIBABA,dlinear,TEST,65.7534,21.288099,31.7280,24.8123,-0.3275
1,AMAZON,dlinear,TEST,52.9680,8.931800,21.8394,19.2556,-0.7190
2,GOOGLE,dlinear,TEST,46.1187,15.389800,39.6716,31.8381,-0.2936
3,VNM,dlinear,TEST,62.3853,4.950600,3601.2967,2983.9441,-0.4852
4,ALIBABA,node,TEST,66.2100,22.210400,33.2727,25.6449,-0.4599
5,AMAZON,node,TEST,55.7078,8.030200,20.4895,16.7840,-0.5130
6,GOOGLE,node,TEST,44.7489,17.368700,41.8184,34.1180,-0.4374
7,VNM,node,TEST,55.0459,5.487700,3947.5605,3225.4346,-0.7846
8,ALIBABA,hybrid,TEST,68.9498,22.577801,33.9296,25.5164,-0.5181
9,AMAZON,hybrid,TEST,60.2740,7.617900,19.2902,16.0050,-0.3411


Đã lưu test_evaluation_summary.csv

[SUMMARY] Successfully tested 0 tickers: []

 ✓ TESTING COMPLETED
[RESULTS] Metrics & predictions lưu tại: ..\..\LOGS\DLINEAR+NODE
[INFO] Predictions stored in predictions_dict: 12 items ready for statistical testing


## Optional: Comparison Charts (visualization after TEST)

In [7]:
print("\n" + "="*80)
print("POST-TEST: Creating comparison charts for each ticker/model pair")
print("Note: These charts use last 5 days of TEST set only")
print("="*80)

# Predefined test_predictions_by_ticker (populated during TEST LOOP)
# For comparison charts after all TEST runs complete
# This showcases 3 individual charts per ticker (one per model)

if hasattr(locals().get('test_predictions_by_ticker', None), 'keys'):
    print(f"\nFound test_predictions_by_ticker with {len(test_predictions_by_ticker)} tickers")
    
    test_model_colors_map = {'dlinear': 'red', 'node': 'cyan', 'hybrid': 'lightgreen'}
    test_model_labels_map = {'dlinear': 'DLinear', 'node': 'NODE', 'hybrid': 'Hybrid'}
    test_total_charts = 0
    
    for ticker in sorted(test_predictions_by_ticker.keys()):
        if ticker not in test_predictions_by_ticker:
            continue
        
        test_val_dates = test_predictions_by_ticker[ticker].get('test_dates')
        test_y_actual = test_predictions_by_ticker[ticker].get('test_actual')
        
        if test_val_dates is None or test_y_actual is None:
            continue
        
        test_plot_dates = pd.to_datetime(test_val_dates)
        test_last_n = 5
        test_plot_dates_last = test_plot_dates[-test_last_n:]
        test_y_actual_last = test_y_actual[-test_last_n:]
        
        for test_model_type in ['dlinear', 'node', 'hybrid']:
            if test_model_type not in test_predictions_by_ticker[ticker]:
                continue
            
            test_preds = test_predictions_by_ticker[ticker][test_model_type]
            test_preds_last = test_preds[-test_last_n:]
            
            # Create individual chart
            test_fig, test_ax = plt.subplots(figsize=(10, 6))
            test_ax.plot(test_plot_dates_last, test_y_actual_last, marker='o', label='Actual',
                        color='black', linewidth=2.5, markersize=8)
            test_ax.plot(test_plot_dates_last, test_preds_last, marker='x', label='Predicted',
                        color=test_model_colors_map[test_model_type], linewidth=2.5, linestyle='--', markersize=10)
            
            test_model_label = test_model_labels_map[test_model_type]
            test_ax.set_title(f'{ticker} TEST: {test_model_label} (Last 5 Days)',
                             fontsize=14, fontweight='bold', pad=15)
            test_ax.set_xlabel('Date', fontsize=12)
            test_ax.set_ylabel('Price (VND)', fontsize=12)
            test_ax.legend(fontsize=11, loc='best')
            test_ax.grid(True, alpha=0.3)
            test_ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
            test_fig.autofmt_xdate()
            
            # Save with TEST tag
            test_save_filename = f"{ticker}_{test_model_type}_test_last5days.png"
            test_save_path = os.path.join(CHART_DIR, test_save_filename)
            test_fig.savefig(test_save_path, bbox_inches='tight', dpi=100)
            test_fig.show()
            test_fig.close()
            
            print(f"  [{test_model_type}] {ticker}: Saved {test_save_filename}")
            test_total_charts += 1
    
    print(f"\n[DONE] Generated {test_total_charts} comparison charts for TEST")
else:
    print("Note: test_predictions_by_ticker not found or empty. Comparison charts skipped.")
    print("This is optional - main TEST results are already generated above.")


POST-TEST: Creating comparison charts for each ticker/model pair
Note: These charts use last 5 days of TEST set only
Note: test_predictions_by_ticker not found or empty. Comparison charts skipped.
This is optional - main TEST results are already generated above.


## Advanced Analysis: Comparison Charts, Radar Plots & Sentiment Impact

In [8]:
print("\n" + "="*80)
print("COMPARISON CHARTS: 3 MODELS ON TEST DATA (DLinear vs NODE vs Hybrid)")
print("="*80)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("\n[STEP] Creating comparison charts for each ticker (3 models)...")

comparison_charts_created = 0
for ticker in tickers_with_data:
    try:
        # Try to load predictions for all 3 models
        predictions_data = {}
        for model_type in ['dlinear', 'node', 'hybrid']:
            pred_file = os.path.join(LOG_DIR, f"{ticker}_{model_type}_predict.csv")
            if os.path.exists(pred_file):
                df_pred = pd.read_csv(pred_file)
                df_pred['Date'] = pd.to_datetime(df_pred['Date'])
                predictions_data[model_type] = df_pred
        
        if len(predictions_data) < 3:
            print(f"  [SKIP] {ticker}: Prediction files not found for all models")
            continue
        
        print(f"\n  Creating comparison chart for {ticker}...")
        
        # Get data from hybrid model (as reference)
        df_hybrid = predictions_data['hybrid'].copy()
        
        # ✅ Extract TEST data only (Type='Predicted')
        df_test_data = df_hybrid[df_hybrid['Type'] == 'Predicted'].copy()
        
        if len(df_test_data) < 3:
            print(f"    [SKIP] {ticker}: Not enough TEST data")
            continue
        
        # Get last 5 days only for visualization
        df_test_last5 = df_test_data.tail(5).copy()
        
        # Filter weekdays only for cleaner visualization
        df_test_last5 = df_test_last5[df_test_last5['Date'].dt.weekday < 5].copy()
        
        if len(df_test_last5) < 3:
            print(f"    [SKIP] {ticker}: Not enough weekday data")
            continue
        
        # Create figure - SIMPLE comparison of 3 models only
        fig, ax = plt.subplots(figsize=(14, 7))
        
        colors = {
            'dlinear': '#FF6B6B',      # Red
            'node': '#4ECDC4',         # Cyan
            'hybrid': '#95E1D3'        # Light green
        }
        labels = {
            'dlinear': 'DLinear (Price Only)',
            'node': 'NODE (Price Only)',
            'hybrid': 'Hybrid (Price + Sentiment)'
        }
        
        # ✅ Plot ACTUAL PRICE (from TEST)
        ax.plot(df_test_last5['Date'], df_test_last5['Actual'], 
               'ko-', linewidth=3, markersize=9, 
               label='Actual Price (TEST)', zorder=5, alpha=0.95)
        
        # ✅ Plot predictions from each of the 3 models
        for model_type in ['dlinear', 'node', 'hybrid']:
            if model_type in predictions_data:
                df_pred = predictions_data[model_type]
                # Get only TEST data (Type='Predicted')
                df_test = df_pred[df_pred['Type'] == 'Predicted'].copy()
                # Filter to last 5 days
                df_test = df_test.tail(5).copy()
                # Filter weekdays
                df_test = df_test[df_test['Date'].dt.weekday < 5].copy()
                
                if len(df_test) > 0:
                    ax.plot(df_test['Date'].values, df_test['Close'].values, 
                           'o-', linewidth=2.5, markersize=7,
                           label=labels[model_type], color=colors[model_type], 
                           alpha=0.85, linestyle='-')
        
        # ✅ Formatting
        ax.set_xlabel('Date', fontsize=12, fontweight='bold')
        ax.set_ylabel('Price (VND)', fontsize=12, fontweight='bold')
        ax.set_title(f'{ticker}: Comparison 3 Models on TEST Data (Last 5 Days)', 
                    fontsize=14, fontweight='bold', pad=15)
        ax.legend(fontsize=11, loc='best', framealpha=0.95)
        ax.grid(True, alpha=0.3, linestyle='--')
        
        # Format x-axis dates
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
        plt.xticks(rotation=45, ha='right', fontsize=11)
        
        plt.tight_layout()
        
        # Save
        comparison_file = os.path.join(CHART_DIR, f"{ticker}_3models_comparison_test.png")
        plt.savefig(comparison_file, bbox_inches='tight', dpi=150)
        print(f"    ✓ Saved: {comparison_file}")
        plt.show()
        plt.close(fig)
        comparison_charts_created += 1
        
    except Exception as e:
        print(f"  [ERROR] {ticker}: {e}")
        import traceback
        traceback.print_exc()
        continue

print(f"\n[DONE] Created {comparison_charts_created} comparison charts")
print("       Each chart compares: DLinear vs NODE vs Hybrid (on TEST data only)\n")



COMPARISON CHARTS: 3 MODELS ON TEST DATA (DLinear vs NODE vs Hybrid)

[STEP] Creating comparison charts for each ticker (3 models)...

[DONE] Created 0 comparison charts
       Each chart compares: DLinear vs NODE vs Hybrid (on TEST data only)



In [9]:
print("\n" + "="*80)
print("MODEL COMPARISON ANALYSIS ON TEST DATA")
print("Compare R², RMSE, MAPE: DLinear vs NODE vs Hybrid")
print("="*80)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────
# LOAD TEST SUMMARY RESULTS
# ─────────────────────────────────────────────────────────────────
print("\n[STEP 1] Loading TEST results...")

# Try to get test_summary from kernel memory, or load from CSV
if 'test_summary' in locals() and test_summary is not None and len(test_summary) > 0:
    print(f"✓ Using test_summary from kernel memory ({len(test_summary)} rows)")
else:
    # Load from CSV file
    summary_csv = os.path.join(LOG_DIR, 'test_evaluation_summary.csv')
    if os.path.exists(summary_csv):
        test_summary = pd.read_csv(summary_csv, encoding='utf-8-sig')
        print(f"✓ Loaded test_summary from CSV ({len(test_summary)} rows)")
    else:
        print("[WARNING] test_summary not found. Skipping analysis.")
        test_summary = None

if test_summary is not None and len(test_summary) > 0:
    print(f"\nFound {len(test_summary)} test results\n")
    
    # Reorganize data by ticker and model type
    results_by_ticker = {}
    for idx, row in test_summary.iterrows():
        ticker = row['Ticker']
        model = row['Model_Type']
        if ticker not in results_by_ticker:
            results_by_ticker[ticker] = {}
        results_by_ticker[ticker][model] = {
            'R2': row.get('R2', 0),
            'RMSE': row.get('RMSE', 0),
            'MAPE': row.get('MAPE (%)', 0),
            'DA': row.get('DA (%)', 0),
            'MAE': row.get('MAE', 0)
        }
    
    # ─────────────────────────────────────────────────────────────────
    # STEP 2: COMPARE 3 MODELS
    # ─────────────────────────────────────────────────────────────────
    print("[STEP 2] Comparing 3 models on TEST data...")
    
    comparison_analysis = []
    
    for ticker in tickers_with_data:
        if ticker not in results_by_ticker:
            continue
        
        ticker_results = results_by_ticker[ticker]
        
        dlinear_results = ticker_results.get('dlinear', {})
        node_results = ticker_results.get('node', {})
        hybrid_results = ticker_results.get('hybrid', {})
        
        if not (dlinear_results and node_results and hybrid_results):
            continue
        
        # Determine BEST model (highest R2)
        r2_scores = {
            'DLinear': dlinear_results.get('R2', -999),
            'NODE': node_results.get('R2', -999),
            'Hybrid': hybrid_results.get('R2', -999)
        }
        best_model = max(r2_scores, key=r2_scores.get)
        
        comparison_analysis.append({
            'Ticker': ticker,
            'DLinear_R2': dlinear_results.get('R2', 0),
            'NODE_R2': node_results.get('R2', 0),
            'Hybrid_R2': hybrid_results.get('R2', 0),
            'DLinear_RMSE': dlinear_results.get('RMSE', 0),
            'NODE_RMSE': node_results.get('RMSE', 0),
            'Hybrid_RMSE': hybrid_results.get('RMSE', 0),
            'DLinear_MAPE': dlinear_results.get('MAPE', 0),
            'NODE_MAPE': node_results.get('MAPE', 0),
            'Hybrid_MAPE': hybrid_results.get('MAPE', 0),
            'Best_Model': best_model,
        })
    
    if comparison_analysis:
        comp_df = pd.DataFrame(comparison_analysis)
        
        # ─────────────────────────────────────────────────────────────
        # Display R2 Comparison
        # ─────────────────────────────────────────────────────────────
        print("\n[R² SCORES ON TEST DATA]")
        print("(Higher R² = Better model)\n")
        display_r2 = comp_df[['Ticker', 'DLinear_R2', 'NODE_R2', 'Hybrid_R2', 'Best_Model']]
        print(display_r2.to_string(index=False))
        
        # ─────────────────────────────────────────────────────────────
        # Display RMSE Comparison
        # ─────────────────────────────────────────────────────────────
        print("\n[RMSE ON TEST DATA]")
        print("(Lower RMSE = Better model)\n")
        display_rmse = comp_df[['Ticker', 'DLinear_RMSE', 'NODE_RMSE', 'Hybrid_RMSE']]
        print(display_rmse.to_string(index=False))
        
        # ─────────────────────────────────────────────────────────────
        # Save comparison
        # ─────────────────────────────────────────────────────────────
        comp_file = os.path.join(LOG_DIR, 'test_models_comparison.csv')
        comp_df.to_csv(comp_file, index=False, encoding='utf-8-sig')
        print(f"\n✓ Saved comparison -> test_models_comparison.csv")
        
        # ─────────────────────────────────────────────────────────────
        # Create comparison bar charts
        # ─────────────────────────────────────────────────────────────
        print("\n[STEP 3] Creating comparison charts...")
        
        if len(comp_df) > 0:
            fig, axes = plt.subplots(2, 1, figsize=(14, 10))
            x_pos = np.arange(len(comp_df))
            width = 0.25
            
            # ✅ Chart 1: R² Comparison (Higher is Better)
            ax1 = axes[0]
            ax1.bar(x_pos - width, comp_df['DLinear_R2'], width, 
                   label='DLinear (Price Only)', color='#FF6B6B', alpha=0.85)
            ax1.bar(x_pos, comp_df['NODE_R2'], width, 
                   label='NODE (Price Only)', color='#4ECDC4', alpha=0.85)
            ax1.bar(x_pos + width, comp_df['Hybrid_R2'], width, 
                   label='Hybrid (Price + Sentiment)', color='#95E1D3', alpha=0.85)
            
            ax1.set_ylabel('R² Score', fontsize=12, fontweight='bold')
            ax1.set_title('Model Comparison on TEST Data: R² Score (Higher = Better)', 
                         fontsize=13, fontweight='bold', pad=15)
            ax1.set_xticks(x_pos)
            ax1.set_xticklabels(comp_df['Ticker'], fontsize=11, fontweight='bold')
            ax1.legend(fontsize=11, loc='best')
            ax1.grid(True, alpha=0.3, axis='y')
            ax1.set_ylim(bottom=0)
            
            # ✅ Chart 2: RMSE Comparison (Lower is Better)
            ax2 = axes[1]
            ax2.bar(x_pos - width, comp_df['DLinear_RMSE'], width, 
                   label='DLinear (Price Only)', color='#FF6B6B', alpha=0.85)
            ax2.bar(x_pos, comp_df['NODE_RMSE'], width, 
                   label='NODE (Price Only)', color='#4ECDC4', alpha=0.85)
            ax2.bar(x_pos + width, comp_df['Hybrid_RMSE'], width, 
                   label='Hybrid (Price + Sentiment)', color='#95E1D3', alpha=0.85)
            
            ax2.set_ylabel('RMSE', fontsize=12, fontweight='bold')
            ax2.set_title('Model Comparison on TEST Data: RMSE (Lower = Better)', 
                         fontsize=13, fontweight='bold', pad=15)
            ax2.set_xticks(x_pos)
            ax2.set_xticklabels(comp_df['Ticker'], fontsize=11, fontweight='bold')
            ax2.legend(fontsize=11, loc='best')
            ax2.grid(True, alpha=0.3, axis='y')
            ax2.set_ylim(bottom=0)
            
            plt.tight_layout()
            
            comparison_chart_file = os.path.join(CHART_DIR, 'test_models_comparison.png')
            plt.savefig(comparison_chart_file, bbox_inches='tight', dpi=150)
            print(f"✓ Saved comparison chart -> test_models_comparison.png")
            plt.show()
            plt.close(fig)
    
    print("\n" + "="*80)
    print("✓ MODEL COMPARISON COMPLETED!")
    print(f"  • Compared 3 models: DLinear vs NODE vs Hybrid")
    print(f"  • Dataset: TEST data only")
    print(f"  • Tickers analyzed: {len(comparison_analysis)}")
    print("="*80)



MODEL COMPARISON ANALYSIS ON TEST DATA
Compare R², RMSE, MAPE: DLinear vs NODE vs Hybrid

[STEP 1] Loading TEST results...
✓ Using test_summary from kernel memory (12 rows)

Found 12 test results

[STEP 2] Comparing 3 models on TEST data...

✓ MODEL COMPARISON COMPLETED!
  • Compared 3 models: DLinear vs NODE vs Hybrid
  • Dataset: TEST data only
  • Tickers analyzed: 0


## ✓ CELL 7: BƯỚC 2 — Chuẩn bị dữ liệu (Tính Residuals theo ngày)

In [10]:
# =============================================================================
# BƯỚC 2: Chuẩn bị dữ liệu - Tính Residuals/Absolute Errors theo ngày
# =============================================================================

print("\n" + "="*80)
print("BƯỚC 2: Tính residuals (sai số từng ngày) cho mỗi model & ticker")
print("="*80)

# Dictionary để lưu errors theo ticker
errors_dict = {}  # {ticker: {'hybrid': array, 'dlinear': array, 'node': array}}

# Lấy danh sách unique tickers từ predictions_dict
unique_tickers = set([key[0] for key in predictions_dict.keys()])
print(f"\nTickers to analyze: {sorted(unique_tickers)}")

for ticker in sorted(unique_tickers):
    print(f"\n>>> Processing {ticker}")
    errors_dict[ticker] = {}
    
    for model_type in MODEL_TYPES:
        key = (ticker, model_type)
        if key not in predictions_dict:
            print(f"  [SKIP] {model_type}: Không có dữ liệu")
            continue
        
        data = predictions_dict[key]
        y_true = data['y_true']
        y_pred = data['y_pred']
        
        # Tính residuals: errors = y_true - y_pred
        residuals = y_true - y_pred
        
        # Tính absolute errors: |y_true - y_pred|
        abs_errors = np.abs(residuals)
        
        # Lưu vào dictionary
        errors_dict[ticker][model_type] = {
            'y_true': y_true,
            'residuals': residuals,
            'abs_errors': abs_errors,
            'mae': np.mean(abs_errors)
        }
        
        print(f"  [{model_type.upper()}] {len(y_true)} ngày | MAE: {errors_dict[ticker][model_type]['mae']:.6f}")

print("\n✓ Residuals đã được tính toán cho tất cả models & tickers")
print(f"  Tổng cộng: {len(unique_tickers)} tickers × {len(MODEL_TYPES)} models = {len(unique_tickers) * len(MODEL_TYPES)} bộ errors")



BƯỚC 2: Tính residuals (sai số từng ngày) cho mỗi model & ticker

Tickers to analyze: ['ALIBABA', 'AMAZON', 'GOOGLE', 'VNM']

>>> Processing ALIBABA
  [DLINEAR] 220 ngày | MAE: 24.812292
  [NODE] 220 ngày | MAE: 25.644869
  [HYBRID] 220 ngày | MAE: 25.516375

>>> Processing AMAZON
  [DLINEAR] 220 ngày | MAE: 19.255615
  [NODE] 220 ngày | MAE: 16.783979
  [HYBRID] 220 ngày | MAE: 16.004951

>>> Processing GOOGLE
  [DLINEAR] 220 ngày | MAE: 31.838144
  [NODE] 220 ngày | MAE: 34.117981
  [HYBRID] 220 ngày | MAE: 30.370207

>>> Processing VNM
  [DLINEAR] 219 ngày | MAE: 2983.944092
  [NODE] 219 ngày | MAE: 3225.434570
  [HYBRID] 219 ngày | MAE: 3805.903076

✓ Residuals đã được tính toán cho tất cả models & tickers
  Tổng cộng: 4 tickers × 3 models = 12 bộ errors


## ✓ CELL 8: BƯỚC 3 — Chạy Kiểm định Thống kê (t-test & Wilcoxon)

In [11]:
# =============================================================================
# BƯỚC 3: Chạy kiểm định thống kê (t-test & Wilcoxon)
# So sánh sai số: Hybrid vs DLinear, Hybrid vs NODE
# =============================================================================
from scipy import stats

print("\n" + "="*80)
print("BƯỚC 3: Chạy kiểm định thống kê (Paired t-test & Wilcoxon test)")
print("="*80)

# List để lưu kết quả kiểm định
statistical_results = []

print("\nChạy 10 cặp so sánh (5 tickers × 2 cặp comparisons):\n")
print(f"{'Ticker':<10} {'So sánh':<25} {'t-stat':<12} {'p-value (t-test)':<18} {'W-stat':<12} {'p-value (Wilcoxon)':<20} {'Kết luận':<30}")
print("="*140)

for ticker in sorted(unique_tickers):
    if ticker not in errors_dict:
        continue
    
    ticker_errors = errors_dict[ticker]
    
    # Kiểm tra xem cả 3 models đều có dữ liệu
    if not all(m in ticker_errors for m in ['hybrid', 'dlinear', 'node']):
        print(f"[SKIP] {ticker}: Thiếu dữ liệu từ một hoặc nhiều models")
        continue
    
    # Lấy absolute errors từng ngày
    abs_errors_hybrid   = ticker_errors['hybrid']['abs_errors']
    abs_errors_dlinear  = ticker_errors['dlinear']['abs_errors']
    abs_errors_node     = ticker_errors['node']['abs_errors']
    
    # ─ So sánh 1: Hybrid vs DLinear ─────────────────────────────────
    if len(abs_errors_hybrid) == len(abs_errors_dlinear):
        t_stat_h_d, p_val_t_h_d = stats.ttest_rel(abs_errors_hybrid, abs_errors_dlinear)
        w_stat_h_d, p_val_w_h_d = stats.wilcoxon(abs_errors_hybrid, abs_errors_dlinear)
        
        # Xác định kết luận
        significant_h_d = "✓ Có ý nghĩa" if p_val_t_h_d < 0.05 else "✗ Không có ý nghĩa"
        
        result_h_d = {
            'Ticker': ticker,
            'So sánh': 'Hybrid vs DLinear',
            't-stat': t_stat_h_d,
            'p-value (t-test)': p_val_t_h_d,
            'W-stat': w_stat_h_d,
            'p-value (Wilcoxon)': p_val_w_h_d,
            'Kết luận': significant_h_d,
            'Hybrid_MAE': ticker_errors['hybrid']['mae'],
            'DLinear_MAE': ticker_errors['dlinear']['mae'],
            'NODE_MAE': ticker_errors['node']['mae']
        }
        statistical_results.append(result_h_d)
        
        print(f"{ticker:<10} {'Hybrid vs DLinear':<25} {t_stat_h_d:<12.4f} {p_val_t_h_d:<18.6f} {w_stat_h_d:<12.4f} {p_val_w_h_d:<20.6f} {significant_h_d:<30}")
    
    # ─ So sánh 2: Hybrid vs NODE ────────────────────────────────────
    if len(abs_errors_hybrid) == len(abs_errors_node):
        t_stat_h_n, p_val_t_h_n = stats.ttest_rel(abs_errors_hybrid, abs_errors_node)
        w_stat_h_n, p_val_w_h_n = stats.wilcoxon(abs_errors_hybrid, abs_errors_node)
        
        # Xác định kết luận
        significant_h_n = "✓ Có ý nghĩa" if p_val_t_h_n < 0.05 else "✗ Không có ý nghĩa"
        
        result_h_n = {
            'Ticker': ticker,
            'So sánh': 'Hybrid vs NODE',
            't-stat': t_stat_h_n,
            'p-value (t-test)': p_val_t_h_n,
            'W-stat': w_stat_h_n,
            'p-value (Wilcoxon)': p_val_w_h_n,
            'Kết luận': significant_h_n,
            'Hybrid_MAE': ticker_errors['hybrid']['mae'],
            'DLinear_MAE': ticker_errors['dlinear']['mae'],
            'NODE_MAE': ticker_errors['node']['mae']
        }
        statistical_results.append(result_h_n)
        
        print(f"{ticker:<10} {'Hybrid vs NODE':<25} {t_stat_h_n:<12.4f} {p_val_t_h_n:<18.6f} {w_stat_h_n:<12.4f} {p_val_w_h_n:<20.6f} {significant_h_n:<30}")

print("\n" + "="*140)
print(f"✓ Kiểm định thống kê hoàn thành. Tổng cộng {len(statistical_results)} cặp so sánh.")



BƯỚC 3: Chạy kiểm định thống kê (Paired t-test & Wilcoxon test)

Chạy 10 cặp so sánh (5 tickers × 2 cặp comparisons):

Ticker     So sánh                   t-stat       p-value (t-test)   W-stat       p-value (Wilcoxon)   Kết luận                      
ALIBABA    Hybrid vs DLinear         1.2509       0.212314           11724.0000   0.648397             ✗ Không có ý nghĩa            
ALIBABA    Hybrid vs NODE            -0.3004      0.764135           11884.0000   0.774332             ✗ Không có ý nghĩa            
AMAZON     Hybrid vs DLinear         -5.8445      0.000000           7120.0000    0.000000             ✓ Có ý nghĩa                  
AMAZON     Hybrid vs NODE            -2.6317      0.009101           9405.0000    0.003621             ✓ Có ý nghĩa                  
GOOGLE     Hybrid vs DLinear         -2.3048      0.022117           9581.0000    0.006464             ✓ Có ý nghĩa                  
GOOGLE     Hybrid vs NODE            -3.4441      0.000687           9175.00

## ✓ CELL 9: BƯỚC 4 — Trình bày kết quả (Bảng kiểm định thống kê)

In [12]:
# =============================================================================
# BƯỚC 4: Trình bày kết quả kiểm định thống kê (Bảng)
# Ngưỡng kết luận: p < 0.05 → sự cải thiện có ý nghĩa thống kê
# =============================================================================

print("\n" + "="*80)
print("BƯỚC 4: BẢNG KẾT QUẢ KIỂM ĐỊNH THỐNG KÊ")
print("="*80)

if statistical_results:
    # Tạo DataFrame
    stats_df = pd.DataFrame(statistical_results)
    
    # Chọn các cột để hiển thị
    display_cols = ['Ticker', 'So sánh', 't-stat', 'p-value (t-test)', 'W-stat', 'p-value (Wilcoxon)', 'Kết luận']
    
    print("\n[BẢNG CHÍNH] Kết quả kiểm định thống kê:")
    print("-" * 140)
    display(stats_df[display_cols])
    
    # Lưu CSV
    csv_path = os.path.join(LOG_DIR, 'statistical_testing_results.csv')
    stats_df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    print(f"\n✓ Kết quả lưu vào: {csv_path}")
    
    # ── Bảng MAE so sánh ────────────────────────────────────────────────────
    print("\n" + "-" * 100)
    print("[BẢNG PHỤ] So sánh MAE (Mean Absolute Error) giữa 3 models:")
    print("-" * 100)
    
    mae_comparison = []
    for ticker in sorted(unique_tickers):
        if ticker in errors_dict and all(m in errors_dict[ticker] for m in ['hybrid', 'dlinear', 'node']):
            mae_comparison.append({
                'Ticker': ticker,
                'Hybrid MAE': errors_dict[ticker]['hybrid']['mae'],
                'DLinear MAE': errors_dict[ticker]['dlinear']['mae'],
                'NODE MAE': errors_dict[ticker]['node']['mae']
            })
    
    if mae_comparison:
        mae_df = pd.DataFrame(mae_comparison)
        display(mae_df.round(6))
        
        mae_csv_path = os.path.join(LOG_DIR, 'mae_comparison.csv')
        mae_df.to_csv(mae_csv_path, index=False, encoding='utf-8-sig')
        print(f"\n✓ MAE comparison lưu vào: {mae_csv_path}")
    
    # ── Tóm tắt kết quả ─────────────────────────────────────────────────────
    print("\n" + "="*80)
    print("[TÓMLƯỢC] Kết luận từ kiểm định thống kê:")
    print("="*80)
    
    # Đếm các trường hợp có ý nghĩa
    significant_count = len([r for r in statistical_results if "Có ý nghĩa" in r['Kết luận']])
    total_tests = len(statistical_results)
    
    print(f"\nTổng cộng {total_tests} cặp so sánh:")
    print(f"  • Có ý nghĩa thống kê (p < 0.05): {significant_count}/{total_tests}")
    print(f"  • Không có ý nghĩa thống kê (p ≥ 0.05): {total_tests - significant_count}/{total_tests}")
    
    # Chi tiết từng so sánh
    print("\n>>> Chi tiết từng Ticker:")
    for ticker in sorted(unique_tickers):
        ticker_results = [r for r in statistical_results if r['Ticker'] == ticker]
        if ticker_results:
            print(f"\n  {ticker}:")
            for result in ticker_results:
                significance = "✓ SỬ DỤNG ĐƯỢC" if "Có ý nghĩa" in result['Kết luận'] else "✗ KO CÓ Y NGHĨA"
                print(f"    • {result['So sánh']:20} p={result['p-value (t-test)']:.6f}  [{significance}]")
    
    print("\n" + "="*80)
    print("💡 GHI CHÚ: Nếu p < 0.05 → Hybrid model cải thiện có ý nghĩa thống kê so với DLinear/NODE")
    print("           Nếu p ≥ 0.05 → Không có sự khác biệt có ý nghĩa giữa các models")
    print("="*80)
else:
    print("[ERROR] Không có dữ liệu kiểm định thống kê!")



BƯỚC 4: BẢNG KẾT QUẢ KIỂM ĐỊNH THỐNG KÊ

[BẢNG CHÍNH] Kết quả kiểm định thống kê:
--------------------------------------------------------------------------------------------------------------------------------------------


,Ticker,So sánh,t-stat,p-value (t-test),W-stat,p-value (Wilcoxon),Kết luận
0,ALIBABA,Hybrid vs DLinear,1.250878,2.123135e-01,11724.0,6.483966e-01,✗ Không có ý nghĩa
1,ALIBABA,Hybrid vs NODE,-0.300429,7.641350e-01,11884.0,7.743317e-01,✗ Không có ý nghĩa
2,AMAZON,Hybrid vs DLinear,-5.844455,1.826762e-08,7120.0,9.987764e-08,✓ Có ý nghĩa
3,AMAZON,Hybrid vs NODE,-2.631675,9.100737e-03,9405.0,3.620603e-03,✓ Có ý nghĩa
4,GOOGLE,Hybrid vs DLinear,-2.304763,2.211661e-02,9581.0,6.464327e-03,✓ Có ý nghĩa
5,GOOGLE,Hybrid vs NODE,-3.444080,6.865257e-04,9175.0,1.617178e-03,✓ Có ý nghĩa
6,VNM,Hybrid vs DLinear,5.933134,1.156161e-08,6996.0,7.519084e-08,✓ Có ý nghĩa
7,VNM,Hybrid vs NODE,2.114013,3.565057e-02,9834.0,1.851249e-02,✓ Có ý nghĩa



✓ Kết quả lưu vào: ..\..\LOGS\DLINEAR+NODE\statistical_testing_results.csv

----------------------------------------------------------------------------------------------------
[BẢNG PHỤ] So sánh MAE (Mean Absolute Error) giữa 3 models:
----------------------------------------------------------------------------------------------------


,Ticker,Hybrid MAE,DLinear MAE,NODE MAE
0,ALIBABA,25.516375,24.812292,25.644869
1,AMAZON,16.004951,19.255615,16.783979
2,GOOGLE,30.370207,31.838144,34.117981
3,VNM,3805.903076,2983.944092,3225.434570



✓ MAE comparison lưu vào: ..\..\LOGS\DLINEAR+NODE\mae_comparison.csv

[TÓMLƯỢC] Kết luận từ kiểm định thống kê:

Tổng cộng 8 cặp so sánh:
  • Có ý nghĩa thống kê (p < 0.05): 6/8
  • Không có ý nghĩa thống kê (p ≥ 0.05): 2/8

>>> Chi tiết từng Ticker:

  ALIBABA:
    • Hybrid vs DLinear    p=0.212314  [✗ KO CÓ Y NGHĨA]
    • Hybrid vs NODE       p=0.764135  [✗ KO CÓ Y NGHĨA]

  AMAZON:
    • Hybrid vs DLinear    p=0.000000  [✓ SỬ DỤNG ĐƯỢC]
    • Hybrid vs NODE       p=0.009101  [✓ SỬ DỤNG ĐƯỢC]

  GOOGLE:
    • Hybrid vs DLinear    p=0.022117  [✓ SỬ DỤNG ĐƯỢC]
    • Hybrid vs NODE       p=0.000687  [✓ SỬ DỤNG ĐƯỢC]

  VNM:
    • Hybrid vs DLinear    p=0.000000  [✓ SỬ DỤNG ĐƯỢC]
    • Hybrid vs NODE       p=0.035651  [✓ SỬ DỤNG ĐƯỢC]

💡 GHI CHÚ: Nếu p < 0.05 → Hybrid model cải thiện có ý nghĩa thống kê so với DLinear/NODE
           Nếu p ≥ 0.05 → Không có sự khác biệt có ý nghĩa giữa các models
